In [ ]:
import pandas as pd
import numpy as np

FILE = "Copy of Master Data_290102026 2 - Copy.xlsx"

df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

# Clean Category
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)


In [ ]:
df = df[df["Category"].isin(["repeater", "stranger"])].copy()


In [ ]:
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]
    

In [ ]:
T120_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

grp = df.groupby("Child Part", sort=False)

parts = grp.agg({
    "effective_daily_demand": "sum",
    "Inventory_25": "first",
    "Minimum Quantity": "first",
    "Cycle Time": "first",
    "Vertical Machines": lambda x: [
        m.strip()
        for m in ",".join(x.dropna().astype(str)).split(",")
        if m.strip() in T120_MACHINES
    ],
    "Category": "first"
}).reset_index()


In [ ]:
parts = parts[parts["Vertical Machines"].map(len) > 0].copy()


In [ ]:
parts.rename(columns={
    "Child Part": "part",
    "effective_daily_demand": "daily_demand",
    "Inventory_25": "inventory",
    "Minimum Quantity": "min_qty",
    "Cycle Time": "cycle_time",
    "Vertical Machines": "machines"
}, inplace=True)


In [ ]:
parts["net_required_qty"] = (
    parts["daily_demand"]
    + parts["min_qty"]
    - parts["inventory"]
).clip(lower=0)


In [ ]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


In [ ]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


In [ ]:
CHANGEOVER = 40
CAPACITY = 1320
TARGET_DAYS = 3

def compute_score(row):
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    prod_time = qty_if_made * row["cycle_time"]

    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)


In [ ]:
dfm["score"] = dfm.apply(compute_score, axis=1)


In [ ]:
selected = []

for m, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(3))

selected = pd.concat(selected).reset_index(drop=True)


In [ ]:
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]


In [ ]:
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)


In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================
FILE = "Copy of Master Data_290102026 2 - Copy.xlsx"

T120_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

CHANGEOVER = 40          # minutes
CAPACITY = 1320          # minutes per machine per day
TARGET_DAYS = 3
MAX_PARTS_PER_MACHINE = 3

# =========================================================
# STEP 1: LOAD & CLEAN DATA
# =========================================================
df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Only Repeater & Stranger
df = df[df["Category"].isin(["repeater", "stranger"])].copy()

# =========================================================
# STEP 2: EFFECTIVE DAILY DEMAND
# =========================================================
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]

# =========================================================
# STEP 3: PART-LEVEL AGGREGATION (NO MACHINES HERE)
# =========================================================
part_level = (
    df.groupby("Child Part", sort=False)
      .agg({
          "effective_daily_demand": "sum",
          "Inventory_25": "first",
          "Minimum Quantity": "first",
          "Cycle Time": "first",
          "Category": "first"
      })
      .reset_index()
)

# =========================================================
# STEP 4: NET REQUIRED QTY (YOUR LOGIC)
# =========================================================
part_level["net_required_qty"] = (
    part_level["effective_daily_demand"]
    + part_level["Minimum Quantity"]
    - part_level["Inventory_25"]
).clip(lower=0)

# =========================================================
# STEP 5: BUILD PART–MACHINE TABLE (REAL DATA GRAIN)
# =========================================================
rows = []

for _, r in df.iterrows():
    machine = str(r["Vertical Machines"]).strip()

    if machine not in T120_MACHINES:
        continue

    p = part_level.loc[
        part_level["Child Part"] == r["Child Part"]
    ].iloc[0]

    rows.append({
        "part": r["Child Part"],
        "category": r["Category"],
        "machine": machine,
        "daily_demand": p["effective_daily_demand"],
        "inventory": p["Inventory_25"],
        "cycle_time": p["Cycle Time"],
        "net_required_qty": p["net_required_qty"]
    })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("❌ No Repeater/Stranger parts found on 120T machines")

print("✅ dfm created:", dfm.shape)

# =========================================================
# STEP 6: SCORE FUNCTION (HUMAN LOGIC)
# =========================================================
def compute_score(row):
    # Inventory pain
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    # Relief potential
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    # Production time
    prod_time = qty_if_made * row["cycle_time"]

    # Setup penalty
    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    # Monopoly penalty
    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)

dfm["score"] = dfm.apply(compute_score, axis=1)

# =========================================================
# STEP 7: SELECT TOP 2–3 PARTS PER MACHINE
# =========================================================
selected = []

for machine, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(MAX_PARTS_PER_MACHINE))

selected = pd.concat(selected).reset_index(drop=True)

# =========================================================
# STEP 8: QUANTITY TO PRODUCE (3-DAY INVENTORY)
# =========================================================
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]

# =========================================================
# FINAL OUTPUT
# =========================================================
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== FINAL DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)


In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# CONFIG
# =========================================================
FILE = "Copy of Master Data_290102026 2 - Copy.xlsx"

# Human-known 120T machines
RAW_120T_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

# Normalize machine names → MP01, MP05, etc.
T120_MACHINES = {m.replace("-", "").upper() for m in RAW_120T_MACHINES}

CHANGEOVER = 40          # minutes
CAPACITY = 1320          # minutes/day
TARGET_DAYS = 3
MAX_PARTS_PER_MACHINE = 3

# =========================================================
# STEP 1: LOAD DATA
# =========================================================
df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

# =========================================================
# STEP 2: CLEAN CATEGORY
# =========================================================
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Keep only Repeater & Stranger
df = df[df["Category"].isin(["repeater", "stranger"])].copy()

# =========================================================
# STEP 3: CLEAN MACHINE NAMES (CRITICAL FIX)
# =========================================================
df["machine_clean"] = (
    df["Vertical Machines"]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", "", regex=True)   # remove spaces/newlines
    .str.replace("-", "", regex=False)     # remove hyphens
)

print("DEBUG → Unique cleaned machines:")
print(sorted(df["machine_clean"].unique()))

# =========================================================
# STEP 4: EFFECTIVE DAILY DEMAND
# =========================================================
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]

# =========================================================
# STEP 5: PART-LEVEL AGGREGATION (NO MACHINES)
# =========================================================
part_level = (
    df.groupby("Child Part", sort=False)
      .agg({
          "effective_daily_demand": "sum",
          "Inventory_25": "first",
          "Minimum Quantity": "first",
          "Cycle Time": "first",
          "Category": "first"
      })
      .reset_index()
)

# =========================================================
# STEP 6: NET REQUIRED QTY (YOUR LOGIC)
# =========================================================
part_level["net_required_qty"] = (
    part_level["effective_daily_demand"]
    + part_level["Minimum Quantity"]
    - part_level["Inventory_25"]
).clip(lower=0)

# =========================================================
# STEP 7: BUILD PART–MACHINE TABLE (ROW-LEVEL, FIXED)
# =========================================================
rows = []

for _, r in df.iterrows():
    machine = r["machine_clean"]

    if machine not in T120_MACHINES:
        continue

    p = part_level.loc[
        part_level["Child Part"] == r["Child Part"]
    ].iloc[0]

    rows.append({
        "part": r["Child Part"],
        "category": r["Category"],
        "machine": machine,  # normalized
        "daily_demand": p["effective_daily_demand"],
        "inventory": p["Inventory_25"],
        "cycle_time": p["Cycle Time"],
        "net_required_qty": p["net_required_qty"]
    })

dfm = pd.DataFrame(rows)

print("\nDEBUG → dfm shape:", dfm.shape)
print(dfm.head())

if dfm.empty:
    raise ValueError(
        "❌ STILL EMPTY: Check machine names printed above. "
        "They must match MP01, MP05, MP10, MP17"
    )

# =========================================================
# STEP 8: SCORE FUNCTION (HUMAN LOGIC)
# =========================================================
def compute_score(row):
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    prod_time = qty_if_made * row["cycle_time"]

    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)

dfm["score"] = dfm.apply(compute_score, axis=1)

# =========================================================
# STEP 9: SELECT MAX 3 PARTS PER MACHINE
# =========================================================
selected = []

for machine, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(MAX_PARTS_PER_MACHINE))

selected = pd.concat(selected).reset_index(drop=True)

# =========================================================
# STEP 10: QUANTITY TO PRODUCE (3-DAY INVENTORY LOGIC)
# =========================================================
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]

# =========================================================
# FINAL OUTPUT
# =========================================================
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== FINAL DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)


In [ ]:
# ========================= CONFIG =========================
TARGET_COVERAGE_DAYS = 3          # exactly what you want
MAX_PARTS_PER_MACHINE = 3         # your floor rule
MAX_HOURS_PER_DAY = 22
CHANGEOVER_MIN = 40 / 60          # hours

# Filter only non-runners
non_runners = df[df['Category'].isin(['Repeater', 'Stranger'])].copy()

# Net required for next 3 days
non_runners['Net_3day'] = (non_runners['Daily Demand'] * TARGET_COVERAGE_DAYS) - non_runners['Inventory_25']  # use your inventory column
non_runners['Net_3day'] = non_runners['Net_3day'].clip(lower=0)

# Lot size = 3 days worth (big lots = fewer changeovers)
non_runners['Lot_Size'] = (non_runners['Daily Demand'] * TARGET_COVERAGE_DAYS).round(0)
non_runners['Prod_Hours'] = (non_runners['Lot_Size'] * non_runners['Cycle Time']) / 3600

# Urgency score (higher = schedule sooner)
non_runners['Coverage_Days'] = non_runners['Inventory_25'] / non_runners['Daily Demand'].replace(0, 1)
non_runners['Urgency'] = (3 - non_runners['Coverage_Days']) * non_runners['Daily Demand']   # critical first

# Sort: most urgent + same colour together
non_runners = non_runners.sort_values(['Urgency', 'Color'], ascending=[False, True])

# ========================= SCHEDULE =========================
machines = ['MP-01', 'MP-05', 'MP-10', 'MP-11', 'MP-17']   # your 120T machines
schedule = []
machine_load = {m: 0.0 for m in machines}
machine_parts_today = {m: [] for m in machines}

for _, part in non_runners.iterrows():
    if part['Net_3day'] <= 0:
        continue
    
    lot = part['Lot_Size']
    prod_h = part['Prod_Hours']
    color = part['Color']
    
    # Try machines in order of least loaded, prefer same colour
    candidates = sorted(machines, key=lambda m: machine_load[m])
    for m in candidates:
        current_parts = len(machine_parts_today[m])
        if current_parts >= MAX_PARTS_PER_MACHINE:
            continue
        
        setup = CHANGEOVER_MIN if (machine_parts_today[m] and 
                                   machine_parts_today[m][-1]['Color'] != color) else 0
        
        total_h = prod_h + setup
        if machine_load[m] + total_h > MAX_HOURS_PER_DAY:
            continue
        
        # Assign!
        machine_load[m] += total_h
        machine_parts_today[m].append({
            'Part No': part['Part No'],
            'Lot_Size': lot,
            'Prod_Hours': prod_h,
            'Setup_Hours': setup,
            'Color': color,
            'Coverage_After': part['Coverage_Days'] + 3
        })
        
        schedule.append({
            'Machine': m,
            'Part No': part['Part No'],
            'Category': part['Category'],
            'Lot_Size': lot,
            'Total_Hours': total_h,
            'Coverage_Days_After': round(part['Coverage_Days'] + 3, 1)
        })
        break   # assigned, move to next part

# ========================= OUTPUT =========================
print("\n=== 3-DAY INVENTORY PLAN (max 2-3 parts/machine) ===")
for m in machines:
    print(f"\n🛠 {m}  (used {machine_load[m]:.1f}/22 h)")
    if machine_parts_today[m]:
        for p in machine_parts_today[m]:
            print(f"   • {p['Part No']}  ({p['Lot_Size']} pcs)  → {p['Total_Hours']:.1f}h  (setup {p['Setup_Hours']*60:.0f}min)  → {p['Coverage_Days_After']} days stock")
    else:
        print("   (idle or only runners)")

print("\nTotal changeovers today:", sum(1 for m in machine_parts_today.values() if len(m) > 1))